In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("preprocessed_data.csv")

In [3]:
df.isnull().sum()

text        0
setiment    0
dtype: int64

In [4]:
df.dropna(inplace=True)

In [5]:
df.sample(5)

,text,setiment
1212,tn mean handset support teliasonera telia navi...,neutral
278,situat coat magazin print paper continu weak,negative
92,temporari lay off affect entir workforc also i...,negative
1873,januari septemb finnlin net sale rose eur mn e...,positive
2019,major breweri increas domest beer sale per cen...,positive


In [6]:
tfidf = TfidfVectorizer(ngram_range=(1,2),min_df=5,max_df=0.7)
le = LabelEncoder()

In [7]:
x = df['text']
y = df['setiment']

In [8]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [9]:
x_train = tfidf.fit_transform(x_train)
x_test = tfidf.transform(x_test)
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [10]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(2018, 3134)
(2018,)
(505, 3134)
(505,)


In [11]:
# feature selection
from sklearn.feature_selection import SelectKBest, chi2
selector = SelectKBest(chi2, k=500)
x_train = selector.fit_transform(x_train, y_train)
x_test = selector.transform(x_test)
print(x_train.shape)
print(x_test.shape)

(2018, 500)
(505, 500)


model selection

In [12]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [13]:
par = {
    'C': [5, 0.1, 1],
    'kernel': ['linear', 'rbf']
}

svc = SVC(probability=True)
grid_svc = GridSearchCV(svc, par, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_svc.fit(x_train, y_train)

print("Best Parameters:", grid_svc.best_params_)
print("Best Score:", grid_svc.best_score_)
y_pred = grid_svc.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters: {'C': 5, 'kernel': 'linear'}
Best Score: 0.7175380193106159
Accuracy: 0.6495049504950495
Precision: 0.6506612398258521
Recall: 0.6495049504950495
F1 Score: 0.6443283523228591
              precision    recall  f1-score   support

           0       0.72      0.67      0.70       162
           1       0.62      0.78      0.69       181
           2       0.61      0.48      0.54       162

    accuracy                           0.65       505
   macro avg       0.65      0.64      0.64       505
weighted avg       0.65      0.65      0.64       505



In [14]:
par = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()
grid_knn = GridSearchCV(knn, par, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_knn.fit(x_train, y_train)

print("Best Parameters:", grid_knn.best_params_)
print("Best Score:", grid_knn.best_score_)
y_pred = grid_knn.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best Parameters: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}
Best Score: 0.5995860256000787
Accuracy: 0.5544554455445545
Precision: 0.5641866935747799
Recall: 0.5544554455445545
F1 Score: 0.5505945727139072
              precision    recall  f1-score   support

           0       0.68      0.51      0.58       162
           1       0.53      0.71      0.61       181
           2       0.48      0.43      0.45       162

    accuracy                           0.55       505
   macro avg       0.57      0.55      0.55       505
weighted avg       0.56      0.55      0.55       505



In [15]:
par = {
    'n_estimators': [200, 300],
    'learning_rate': [0.1, 0.01],
    'max_depth': [5, 7]
}

xgb = XGBClassifier(objective='multi:softmax', num_class=3)
grid_xgb = GridSearchCV(xgb, par, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_xgb.fit(x_train, y_train)

print("Best Parameters:", grid_xgb.best_params_)
print("Best Score:", grid_xgb.best_score_)
y_pred = grid_xgb.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300}
Best Score: 0.6709579146500259
Accuracy: 0.6277227722772277
Precision: 0.6385626328317486
Recall: 0.6277227722772277
F1 Score: 0.6277403248949397
              precision    recall  f1-score   support

           0       0.75      0.59      0.66       162
           1       0.60      0.72      0.66       181
           2       0.57      0.56      0.56       162

    accuracy                           0.63       505
   macro avg       0.64      0.62      0.63       505
weighted avg       0.64      0.63      0.63       505



In [16]:
pra = {
    'alpha': [1.0, 5.0, 10.0],
    'fit_prior': [True, False]
}
nb = MultinomialNB()
grid_nb = GridSearchCV(nb, pra, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_nb.fit(x_train, y_train)

print("Best Parameters:", grid_nb.best_params_)
print("Best Score:", grid_nb.best_score_)
y_pred = grid_nb.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters: {'alpha': 1.0, 'fit_prior': True}
Best Score: 0.683840257474879
Accuracy: 0.6158415841584158
Precision: 0.6127442457610559
Recall: 0.6158415841584158
F1 Score: 0.6136073310545457
              precision    recall  f1-score   support

           0       0.69      0.72      0.70       162
           1       0.60      0.63      0.62       181
           2       0.55      0.49      0.52       162

    accuracy                           0.62       505
   macro avg       0.61      0.62      0.61       505
weighted avg       0.61      0.62      0.61       505



In [17]:
# best model is svc

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best Parameters: {'C': 5, 'kernel': 'linear'}
Best Score: 0.7175380193106159
Accuracy: 0.6495049504950495
Precision: 0.6506612398258521
Recall: 0.6495049504950495
F1 Score: 0.6443283523228591
              precision    recall  f1-score   support

           0       0.72      0.67      0.70       162
           1       0.62      0.78      0.69       181
           2       0.61      0.48      0.54       162

    accuracy                           0.65       505
   macro avg       0.65      0.64      0.64       505
weighted avg       0.65      0.65      0.64       505